- Bronze -> Silver for `external_reference` (vendor .dat feed, 19 rows total
 across 3 days). Bronze already quarantined malformed rows.

In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "reference"
BRONZE_SOURCE_NAME = "external_reference"
business_date_str = date.today().isoformat()

In [0]:
bronze_df = read_bronze(spark, BRONZE_SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

Bronze row count: 19
root
 |-- ref_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- asset_name: string (nullable = true)
 |-- identifier: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _source_file_record_count_mismatch: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



### 1. Type casting

In [0]:
typed_df = (
    bronze_df
    .withColumn("asset_id", F.trim(F.col("asset_id")))          # external code, e.g. AST001
    .withColumn("asset_name", F.trim(F.col("asset_name")))
    .withColumn("asset_type", F.upper(F.trim(F.col("asset_type"))))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    .withColumn("business_date", F.to_date(F.col("timestamp")))
)

### 2. Crosswalk join
New instruments (AST007) will simply have no PRIOR crosswalk history
to compare.


In [0]:
asset_xwalk = read_crosswalk(spark, "asset")

ASSET_XWALK_EXTERNAL_COL = "external_asset_id"
ASSET_XWALK_INTERNAL_COL = "internal_company_id"   # confirmed via Azure SQL - asset_id_mapping's real column name

asset_xwalk_slim = asset_xwalk.select(
    F.col(ASSET_XWALK_EXTERNAL_COL).alias("asset_id"),
    F.col(ASSET_XWALK_INTERNAL_COL).alias("internal_asset_id"),
)

joined_df = typed_df.join(asset_xwalk_slim, on="asset_id", how="left")

xwalk_miss_df = joined_df.filter(F.col("internal_asset_id").isNull()) \
    .withColumn("reason_code", F.lit("UNKNOWN_CROSSWALK_MAPPING"))
xwalk_miss_count = xwalk_miss_df.count()
if xwalk_miss_count > 0:
    write_quarantine(xwalk_miss_df, SOURCE_NAME)
    print(f"WARNING: {xwalk_miss_count} rows failed crosswalk lookup.")
    print("NOTE: if AST007 (new Day-3 instrument) shows up here, its crosswalk")
    print("row needs adding by whoever owns staging.asset_id_mapping - a new")
    print("instrument still needs an entry in the mapping table, it's just new.")

crosswalked_df = joined_df.filter(F.col("internal_asset_id").isNotNull())

### 3. Day-over-day change detection
Compares each asset's currency/status/asset_type to its own previous
business_date. AST005's EUR -> GBP (Day 2) -> EUR (Day 3) should
surface as `changed = True` on both Day 2 and Day 3 rows.

In [0]:
changed_df = flag_day_over_day_change(
    crosswalked_df,
    key_cols=["internal_asset_id"],
    business_date_col="business_date",
    compare_cols=["currency", "status", "asset_type"],
)

break_rows = changed_df.filter(F.col("changed") == True)
break_count = break_rows.count()
if break_count > 0:
    write_quarantine(break_rows.withColumn("reason_code", F.lit("REFERENCE_BREAK")), SOURCE_NAME)
    print(f"REFERENCE_BREAK: {break_count} rows - expect at least AST005's Day-2 and Day-3 rows.")

REFERENCE_BREAK: 2 rows - expect at least AST005's Day-2 and Day-3 rows.


### 4. Duplicate check
No duplicate scenario is seeded for REFERENCE -
still run the check for consistency across all sources.

In [0]:
KEY_COLS = ["internal_asset_id", "business_date"]
COMPARE_COLS = ["currency", "status", "asset_type"]

deduped_df, duplicates_df, _ = split_duplicates(changed_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)

### 5. Write to Silver
Note: this keeps the LATEST value as current (simple flag, not SCD
Type 2 history) 

In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")
new_instruments = deduped_df.filter(F.col("_prev_currency").isNull()).select("internal_asset_id", "business_date").distinct()
print("New instruments (no prior day to compare against):")
new_instruments.show()

Silver row count: 19
New instruments (no prior day to compare against):
+-----------------+-------------+
|internal_asset_id|business_date|
+-----------------+-------------+
|        PORT_0001|   2026-09-15|
|        PORT_0002|   2026-09-15|
|        PORT_0003|   2026-09-15|
|        PORT_0004|   2026-09-15|
|        PORT_0005|   2026-09-15|
|        PORT_0006|   2026-09-15|
|        PORT_0007|   2026-09-17|
+-----------------+-------------+



In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "crosswalk_miss", joined_df.count(), xwalk_miss_count, "UNKNOWN_CROSSWALK_MAPPING")
log_dq(spark, SOURCE_NAME, business_date_str, "reference_break", crosswalked_df.count(), break_count, "REFERENCE_BREAK")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", changed_df.count(), dup_count, "DUPLICATE_RECORD")

/home/spark-0f80f9b4-a058-4360-91d6-c9/.ipykernel/71/command-5696143635338729-2886423099:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = xwalk_miss_count + dup_count   # break rows are flagged, not removed - see Step 3 note
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=19 = silver=19 + quarantined=0
